Construction of ESG scores

TF/IDF as main measure, mean TF/IDF score as very good variable

Work done on E/G primary indicator words

Normalized search volume for that year is another variable. A high environmental discourse score in a year with high environmental attention could be optics or genuine concern.

How should S be introduced? A stewardship variable where firms are given better scores for consistently beating scores is very overfit. For the sake of clear-cut analysis we drop it.


In [ ]:
import polars as pl

# --------------------
# Load raw parquet
# --------------------
df_raw = (
    pl.read_parquet("spy_10k_2015_present.parquet")
    .with_columns(
        pl.col("filing_date").dt.year().alias("year")
    )
)

import polars as pl

# --------------------
# Load raw parquet
# --------------------
df_raw = (
    pl.read_parquet("spy_10k_2015_present.parquet")
    .with_columns(
        pl.col("filing_date").dt.year().alias("year")
    )
)

# --------------------
# Mean TF-IDF per firm-year
# --------------------
docs_work = (
    df_raw
    .group_by(["cik", "gics_sector", "year"])
    .agg([
        pl.col("tfidf_E").mean().alias("tfidf_E"),
        pl.col("tfidf_G").mean().alias("tfidf_G")
    ])
)

print(docs_work.head(10))
print("Rows:", docs_work.height)

# --------------------
# Mean TF-IDF per firm-year
# --------------------
docs_work = (
    df_raw
    .group_by(["cik", "gics_sector", "year"])
    .agg([
        pl.col("tfidf_E").mean().alias("tfidf_E"),
        pl.col("tfidf_G").mean().alias("tfidf_G")
    ])
)

print(docs_work.head(10))
print("Rows:", docs_work.height)

# --------------------
# Hardcoded annual trends
# --------------------
df_trend = pl.DataFrame({
    "year": [
        2015,2016,2017,2018,2019,
        2020,2021,2022,2023,2024
    ],
    "trend_E": [
        31.08,31.00,32.72,33.37,36.90,
        34.67,30.40,38.58,40.10,40.72
    ],
    "trend_G": [
        44.6,42.8,44.1,42.2,41.6,
        40.8,37.2,44.2,44.5,46.2
    ]
})

# --------------------
# ESG score construction
# --------------------
df_esg = (
    docs_work
    .join(df_trend, on="year", how="left")
    .with_columns([
        (pl.col("tfidf_E") * pl.col("trend_E")).alias("E_score"),
        (pl.col("tfidf_G") * pl.col("trend_G")).alias("G_score")
    ])
    .select([
        "cik",
        "gics_sector",
        "E_score",
        "G_score"
    ])
    .sort("cik")
)

# --------------------
# Output
# --------------------
print(df_esg.head(20))
print("Rows:", df_esg.height)

ColumnNotFoundError: unable to find column "tfidf_E"; valid columns: ["cik", "ticker", "company_name", "form_type", "section", "text", "filing_date", "filing_period", "source_path", "gics_sector", "gics_sub_industry", "added_sp500_date", "founded", "year"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
DF ["cik", "ticker", "company_name", "form_type", ...]; PROJECT */14 COLUMNS